# Football Brothers – Player Similarity Model

This notebook builds a simple player similarity model using Kaggle football datasets (21/22 and 22/23 seasons).


Links:
https://www.kaggle.com/datasets/vivovinco/20212022-football-player-stats/data,
https://www.kaggle.com/datasets/vivovinco/20222023-football-player-stats/data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

sys.path.append(str(Path("..").resolve()))

from src.data_prepare import prepare_base_df, add_per90

DATA_DIR = Path("..") / "data"
MIN_MINUTES = 900

df_base = prepare_base_df(DATA_DIR, min_minutes=MIN_MINUTES, drop_gk=True)

df_base.shape, df_base.columns[:]

((2511, 123),
 Index(['2CrdY', '90s', 'AerLost', 'AerWon', 'AerWon%', 'Age', 'Assists',
        'BlkPass', 'BlkSh', 'Blocks',
        ...
        'TouAtt3rd', 'TouAttPen', 'TouDef3rd', 'TouDefPen', 'TouLive',
        'TouMid3rd', 'Touches', 'Pos_tags', 'Is_hybrid', 'Pos_group'],
       dtype='str', length=123))

## Feature Grouping Strategy

Players are compared using grouped football metrics.
Each group has a custom weight.

In [2]:
FEATURE_GROUPS = {
    # 1) shooting threat
    "threat": {
        "weight": 0.18,
        "per90": ["Goals", "Shots", "SoT"],
        "raw": ["SoT%", "G/Sh", "ShoDist"],
    },

    # 2) Creation
    "creation": {
        "weight": 0.15,
        "per90": ["Assists", "PasAss", "SCA", "GCA", "PPA", "CrsPA"],
        "raw": [],
    },

    # 3A) passing progression
    "progression_pass": {
        "weight": 0.10,
        "per90": ["PasProg", "Pas3rd", "TB", "Sw", "PasTotPrgDist"],
        "raw": [],
    },

    # 3B) carry progression
    "progression_carry": {
        "weight": 0.08,
        "per90": ["CarProg", "CarPrgDist", "CPA", "RecProg"],
        "raw": [],
    },

    # 4A) passing volume
    "passing_volume": {
        "weight": 0.09,
        "per90": ["PasTotAtt", "PasTotCmp", "PasTotDist"],
        "raw": [],
    },

    # 4B) quality of passes
    "passing_quality": {
        "weight": 0.05,
        "per90": [],
        "raw": ["PasTotCmp%"],
    },

    # 5) Touch profiles - role
    "touch_profile": {
        "weight": 0.10,
        "per90": ["Touches", "TouDef3rd", "TouMid3rd", "TouAtt3rd", "TouAttPen"],
        "raw": [],
    },

    # 6) Defensywa
    "defense_actions": {
        "weight": 0.15,
        "per90": ["Tkl", "Int", "Tkl+Int", "Blocks", "Clr"],
        "raw": [],
    },

    # 7) Aerial duels
    "aerial": {
        "weight": 0.06,
        "per90": ["AerWon", "AerLost"],
        "raw": ["AerWon%"],
    },

    # 8) Ball security
    "ball_security": {
        "weight": 0.04,
        "per90": ["CarMis", "CarDis"],
        "raw": [],
    },
}

In [3]:
per90_cols = sorted({c for g in FEATURE_GROUPS.values() for c in g["per90"]})
raw_cols  = sorted({c for g in FEATURE_GROUPS.values() for c in g["raw"]})

missing = [c for c in (per90_cols + raw_cols) if c not in df_base.columns]
print("Missing features:", missing)

per90_cols = [c for c in per90_cols if c in df_base.columns]
raw_cols = [c for c in raw_cols if c in df_base.columns]

print("Using per90:", len(per90_cols), "Using raw:", len(raw_cols))

Missing features: []
Using per90: 35 Using raw: 5


## Feature Matrix Construction

Selected features are transformed to per90 metrics where needed.
Final matrix is built using weighted feature groups.

In [4]:
for c in raw_cols:
    df_base[c] = pd.to_numeric(df_base[c], errors="coerce")

df_base = add_per90(df_base, per90_cols)

feature_cols_final = [c + "_per90" for c in per90_cols] + raw_cols
X = df_base[feature_cols_final].copy()

print("X shape:", X.shape)
X.isna().mean().sort_values(ascending=False).head(10)

X shape: (2511, 40)


AerLost_per90       0.0
AerWon_per90        0.0
Assists_per90       0.0
Blocks_per90        0.0
CPA_per90           0.0
CarDis_per90        0.0
CarMis_per90        0.0
CarPrgDist_per90    0.0
CarProg_per90       0.0
Clr_per90           0.0
dtype: float64

In [5]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

imputer = SimpleImputer(strategy="median")
scaler = RobustScaler()

X_imp = imputer.fit_transform(X)
X_scaled = scaler.fit_transform(X_imp)

X_scaled.shape

(2511, 40)

In [6]:
import numpy as np

X_weighted = X_scaled.copy()
col_index = {col: i for i, col in enumerate(feature_cols_final)}

for group_name, cfg in FEATURE_GROUPS.items():
    w = cfg["weight"]
    cols = [c + "_per90" for c in cfg["per90"]] + cfg["raw"]
    cols = [c for c in cols if c in feature_cols_final]
    factor = np.sqrt(w)
    for c in cols:
        X_weighted[:, col_index[c]] *= factor

X_weighted.shape

(2511, 40)

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity(X_weighted)
sim.shape

(2511, 2511)

## Similarity Modeling

Cosine similarity is used to measure stylistic similarity between players.
It focuses on the direction of a player’s statistical profile rather than the absolute magnitude of the values.

In [8]:
def find_similar(player_name: str, season: str, top_n: int = 100, power: int = 2 ):
    mask = (df_base["Player"] == player_name) & (df_base["Season"] == season)
    if mask.sum() == 0:
        raise ValueError("Not found player + season")

    idx = df_base.index[mask][0]
    idx_pos = df_base.index.get_loc(idx)

    scores = sim[idx_pos].copy()

    # don't show yourself
    scores[idx_pos] = -1

    # don't show different season of same player
    same_player = (df_base["Player"].values == player_name)
    scores[same_player] = -1

    # calibration for better difference in similarity
    scores_cal = scores.copy()
    pos = scores_cal > 0
    scores_cal[pos] = scores_cal[pos] ** power

    top_idx = np.argsort(scores_cal)[::-1][:top_n]

    result = df_base.iloc[top_idx][
        ["Player", "Season", "Squad", "Comp", "Pos", "Min"]
    ].copy()

    result["similarity"] = scores_cal[top_idx].round(3)

    return result.sort_values("similarity", ascending=False)

In [9]:
df_base[["Player","Season"]].head(100)

,Player,Season
0,Max Aarons,2021/22
1,Yunis Abdelhamid,2021/22
2,Salis Abdul Samed,2021/22
3,Laurent Abergel,2021/22
4,Tammy Abraham,2021/22
...,...,...
95,Stéphane Bahoken,2021/22
96,Nedim Bajrami,2021/22
97,Mitchel Bakker,2021/22
98,Ridle Baku,2021/22


In [10]:
find_similar("Nico Williams", "2021/22", top_n=10)

,Player,Season,Squad,Comp,Pos,Min,similarity
1389,Adama Traoré,2021/22,Wolves,Premier League,FW,1067,0.804
769,Takefusa Kubo,2021/22,Mallorca,La Liga,FWMF,1605,0.778
547,Jack Grealish,2021/22,Manchester City,Premier League,FW,1914,0.724
1131,Marko Pjaca,2021/22,Torino,Serie A,MFFW,1149,0.704
1586,Lameck Banda,2022/23,Lecce,Serie A,FWMF,1051,0.692
41,Jim Allevinah,2021/22,Clermont Foot,Ligue 1,FW,1488,0.688
639,Callum Hudson-Odoi,2021/22,Chelsea,Premier League,FWMF,962,0.688
653,Jonathan Ikone,2021/22,Lille,Ligue 1,MFFW,1062,0.684
1311,Riccardo Sottil,2021/22,Fiorentina,Serie A,FW,1239,0.674
1677,Yannick Carrasco,2022/23,Atlético Madrid,La Liga,DFMF,969,0.665


### Example: Nico Williams (2021/22)

The model returns mostly attacking wingers and wide forwards such as Adama Traoré, Takefusa Kubo or Jack Grealish.
These players share a similar profile based on ball progression, carries and attacking involvement in wide areas.

The results look reasonable, as Nico Williams is primarily a direct winger focused on dribbling, progression and attacking contribution from the flanks.

In [11]:
def compare_players(base_player: str, base_season: str, top_n: int = 5):
    mask = (df_base["Player"] == base_player) & (df_base["Season"] == base_season)
    if mask.sum() == 0:
        raise ValueError("Nie znaleziono takiego Player + Season.")

    base_idx = df_base.index[mask][0]
    base_pos = df_base.index.get_loc(base_idx)

    scores = sim[base_pos].copy()

    scores[base_pos] = -1
    same_player = (df_base["Player"].values == base_player)
    scores[same_player] = -1

    top_idx_pos = np.argsort(scores)[::-1][:top_n]
    top_indices = df_base.index[top_idx_pos]


    selected_rows = df_base.loc[[base_idx] + list(top_indices)]
    stats_table = selected_rows[["Player", "Season"] + feature_cols_final].copy()

    return stats_table.round(3)

In [12]:
compare_players("Nico Williams", "2021/22", top_n=5)

,Player,Season,AerLost_per90,AerWon_per90,Assists_per90,Blocks_per90,CPA_per90,CarDis_per90,CarMis_per90,CarPrgDist_per90,...,TouAtt3rd_per90,TouAttPen_per90,TouDef3rd_per90,TouMid3rd_per90,Touches_per90,AerWon%,G/Sh,PasTotCmp%,ShoDist,SoT%
1470,Nico Williams,2021/22,0.068,0.050,0.000,0.108,0.140,0.144,0.171,10.007,...,1.872,0.397,0.374,1.148,3.154,42.3,0.00,71.2,15.5,30.3
1389,Adama Traoré,2021/22,0.183,0.085,0.000,0.056,0.205,0.247,0.261,16.076,...,1.966,0.396,0.403,1.639,3.513,31.6,0.04,74.5,18.9,26.1
769,Takefusa Kubo,2021/22,0.057,0.041,0.000,0.101,0.072,0.133,0.133,7.567,...,1.270,0.234,0.363,1.107,2.511,41.9,0.02,70.3,19.8,18.4
547,Jack Grealish,2021/22,0.066,0.035,0.007,0.044,0.194,0.095,0.057,10.310,...,1.746,0.428,0.163,1.014,2.681,34.8,0.07,86.5,14.3,35.6
1131,Marko Pjaca,2021/22,0.226,0.122,0.000,0.080,0.116,0.213,0.189,8.484,...,1.914,0.342,0.262,1.648,3.547,35.1,0.14,77.2,16.3,42.9
1586,Lameck Banda,2022/23,0.117,0.037,0.008,0.132,0.168,0.161,0.278,10.145,...,1.769,0.329,0.373,1.239,3.282,23.8,0.05,62.6,19.5,33.3


## Some examples


In [13]:
find_similar("Erling Haaland", "2022/23", top_n=8)

,Player,Season,Squad,Comp,Pos,Min,similarity
1979,Harry Kane,2022/23,Tottenham,Premier League,FW,2055,0.984
2206,Victor Osimhen,2022/23,Napoli,Serie A,FW,1401,0.984
2152,Terem Moffi,2022/23,Lorient,Ligue 1,FW,1386,0.979
1584,Folarin Balogun,2022/23,Reims,Ligue 1,FW,1591,0.979
2019,Alexandre Lacazette,2022/23,Lyon,Ligue 1,FW,1923,0.978
2103,Lautaro Martínez,2022/23,Inter,Serie A,FW,1675,0.975
1804,Breel Embolo,2022/23,Monaco,Ligue 1,FW,1542,0.973
2427,Ivan Toney,2022/23,Brentford,Premier League,FW,1789,0.973


### Erling Haaland (2022/23)

The model returns mostly classic strikers such as Harry Kane, Victor Osimhen or Ivan Toney.
This matches Haaland’s profile as a goal-focused centre-forward with strong scoring and finishing statistics.

In [14]:
find_similar("Memphis Depay", "2021/22", top_n=8)

,Player,Season,Squad,Comp,Pos,Min,similarity
952,Dries Mertens,2021/22,Napoli,Serie A,FWMF,1384,0.818
467,Phil Foden,2021/22,Manchester City,Premier League,FW,2128,0.788
549,Mason Greenwood,2021/22,Manchester Utd,Premier League,FWMF,1271,0.783
1407,Cengiz Ünder,2021/22,Marseille,Ligue 1,FWMF,2200,0.757
555,Arnaut Groeneveld,2021/22,Villarreal,La Liga,FW,1472,0.752
116,Musa Barrow,2021/22,Bologna,Serie A,FWMF,2134,0.750
869,Donyell Malen,2021/22,Dortmund,Bundesliga,FW,1684,0.737
861,Riyad Mahrez,2021/22,Manchester City,Premier League,FW,1498,0.733


### Example: Memphis Depay (2021/22)

The model returns mainly hybrid forwards and technical attacking players such as Dries Mertens, Phil Foden or Riyad Mahre.
This makes sense, as Depay’s profile is not that of a classic striker, but rather a mobile forward combining scoring, ball progression and creative involvement.

In [15]:
find_similar("João Félix", "2021/22", top_n=8)

,Player,Season,Squad,Comp,Pos,Min,similarity
987,Gerard Moreno,2021/22,Villarreal,La Liga,FW,1191,0.852
1003,Luis Muriel,2021/22,Atalanta,Serie A,FW,1539,0.766
1476,Florian Wirtz,2021/22,Leverkusen,Bundesliga,MFFW,1851,0.763
1029,Neymar,2021/22,Paris S-G,Ligue 1,FWMF,1851,0.763
1335,Isaac Success,2021/22,Udinese,Serie A,FW,911,0.759
652,Kelechi Iheanacho,2021/22,Leicester City,Premier League,FW,1264,0.745
1244,Oihan Sancet,2021/22,Athletic Club,La Liga,FW,1422,0.736
363,Ángel Di María,2021/22,Paris S-G,Ligue 1,FW,1645,0.734


### Example: João Félix (2021/22)

João Félix's profile is defined more by link-up play, creativity and high dribbling involvement rather than pure striker or winger output. Because of his hybrid attacking role, his profile is harder to categorize, which is reflected in the results where we can see a mix of attacking midfielders, creative strikers and wingers.

In [16]:
find_similar("Kevin De Bruyne", "2021/22", top_n=8)

,Player,Season,Squad,Comp,Pos,Min,similarity
870,Ruslan Malinovskyi,2021/22,Atalanta,Serie A,FWMF,1592,0.823
363,Ángel Di María,2021/22,Paris S-G,Ligue 1,FW,1645,0.820
1029,Neymar,2021/22,Paris S-G,Ligue 1,FWMF,1851,0.793
987,Gerard Moreno,2021/22,Villarreal,La Liga,FW,1191,0.762
577,Ýlkay Gündoðan,2021/22,Manchester City,Premier League,MF,1857,0.744
1135,Daniel Podence,2021/22,Wolves,Premier League,FWMF,1484,0.737
657,Lorenzo Insigne,2021/22,Napoli,Serie A,FW,2290,0.731
629,Jonas Hofmann,2021/22,M'Gladbach,Bundesliga,MFFW,2078,0.730


### Kevin De Bruyne (2021/22)

The model returns mostly creative attacking players such as Ruslan Malinovskyi, Ángel Di María or Neymar, which fits De Bruyne's profile as a highly creative attacking midfielder.

Interestingly, Gerard Moreno (in season 2021/22 – 24 games as striker) appears again among similar players. Even though he played mainly as a striker, his statistical profile includes a lot of chance creation and creative involvement, which makes him comparable to more creative roles.

In some examples the list of similar players was easy to interpret, but in other cases it was less obvious why the model linked certain players together. To help with this, I added a helper function called explain_similarity().

This function compares the base player with the returned similar players and shows the features with the smallest differences between them after scaling. These are the statistics where the players are most similar.

It is important to note that this does not mean these features have the biggest impact on the similarity score. In defensive roles, attacking metrics such as goals or assists may appear simply because a lot of players have similarly low values in them.

In [17]:
def explain_similarity(base_player: str, base_season: str, top_n: int = 8, n_features: int = 5):
    df_compare = compare_players(base_player, base_season, top_n=top_n).reset_index(drop=True)

    meta_cols = ["Player", "Season"]
    feature_cols = [c for c in df_compare.columns if c not in meta_cols]
    X = df_compare[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

    scaler = RobustScaler()
    X_scaled = pd.DataFrame(
        scaler.fit_transform(X),
        columns=feature_cols
    )
    base_vec = X_scaled.iloc[0]
    rows = []

    for i in range(1, len(df_compare)):
        player = df_compare.loc[i, "Player"]
        diff = (base_vec - X_scaled.iloc[i]).abs().sort_values()

        rows.append({
            "player": player,
            "most_similar_features": ", ".join(diff.index[:n_features])
        })

    return pd.DataFrame(rows)

In [18]:
from sklearn.preprocessing import RobustScaler
import pandas as pd

def compare_and_explain(base_player: str, base_season: str, top_n: int = 8, n_features: int = 5):
    df_compare = compare_players(base_player, base_season, top_n=top_n).reset_index(drop=True)

    meta_cols = ["Player", "Season"]
    feature_cols = [c for c in df_compare.columns if c not in meta_cols]

    X = df_compare[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

    scaler = RobustScaler()
    X_scaled = pd.DataFrame(
        scaler.fit_transform(X),
        columns=feature_cols
    )

    base_vec = X_scaled.iloc[0]
    rows = []

    for i in range(1, len(df_compare)):
        player = df_compare.loc[i, "Player"]
        diff = (base_vec - X_scaled.iloc[i]).abs().sort_values()

        rows.append({
            "player": player,
            "most_similar_features": ", ".join(diff.index[:n_features])
        })

    explain_df = pd.DataFrame(rows)

    return df_compare, explain_df

In [19]:
find_similar("Rodri", "2022/23", top_n=8)

,Player,Season,Squad,Comp,Pos,Min,similarity
2013,Toni Kroos,2022/23,Real Madrid,La Liga,MF,1219,0.790
2008,Jules Koundé,2022/23,Barcelona,La Liga,DF,1031,0.762
2109,Nemanja Mati?,2022/23,Roma,Serie A,MF,1223,0.727
2392,Niklas Süle,2022/23,Dortmund,Bundesliga,DF,1148,0.720
2504,Oleksandr Zinchenko,2022/23,Arsenal,Premier League,DF,1081,0.710
2386,John Stones,2022/23,Manchester City,Premier League,DF,1181,0.702
2230,Danilo Pereira,2022/23,Paris S-G,Ligue 1,DFMF,1184,0.686
1595,Alessandro Bastoni,2022/23,Inter,Serie A,DF,1267,0.682


In [20]:
explain_similarity("Rodri", "2022/23", top_n=8, n_features=8)

,player,most_similar_features
0,Toni Kroos,"Assists_per90, PasTotCmp%, CarMis_per90, CarDi..."
1,Jules Koundé,"PPA_per90, Tkl+Int_per90, Tkl_per90, Sw_per90,..."
2,Nemanja Mati?,"TouMid3rd_per90, RecProg_per90, Sw_per90, TouD..."
3,Niklas Süle,"GCA_per90, ShoDist, TouMid3rd_per90, Shots_per..."
4,Oleksandr Zinchenko,"Int_per90, SCA_per90, SoT%, PasAss_per90, SoT_..."
5,John Stones,"Assists_per90, CarProg_per90, PasProg_per90, A..."
6,Danilo Pereira,"PasProg_per90, CarDis_per90, PasTotDist_per90,..."
7,Alessandro Bastoni,"Touches_per90, PasTotAtt_per90, PasProg_per90,..."


### Rodri (2022/23)

For Rodri, we would expect a deep-lying midfielder profile built around passing volume, progression and positional control in possession.

The results support this idea, but in a more interesting way than before. Toni Kroos appears as a very natural match, since both players are heavily involved in ball movement, tempo control and moving the ball forward from deeper areas. At the same time, the presence of players such as John Stones, Bastoni, Koundé or even Zinchenko suggests that the model does not see Rodri only as a classic defensive midfielder.

Instead, his profile overlaps with defenders who play an important role in build-up and controlled progression. This makes Rodri look less like a traditional ball-winning midfielder and more like a deep possession hub whose statistical profile combines midfield control with traits often associated with ball-playing defenders.


In [21]:
find_similar("Nampalys Mendy", "2021/22", top_n=8)

,Player,Season,Squad,Comp,Pos,Min,similarity
1846,Idrissa Gana Gueye,2022/23,Everton,Premier League,MF,1221,0.655
503,Eric García,2021/22,Barcelona,La Liga,DF,2039,0.632
156,Juan Bernat,2021/22,Paris S-G,Ligue 1,DF,958,0.599
144,Nabil Bentaleb,2021/22,Angers,Ligue 1,MF,1515,0.510
50,Sofyan Amrabat,2021/22,Fiorentina,Serie A,MF,944,0.493
263,Trevoh Chalobah,2021/22,Chelsea,Premier League,DF,1449,0.476
169,Jérôme Boateng,2021/22,Lyon,Ligue 1,DF,1622,0.432
1548,Daniel Amartey,2022/23,Leicester City,Premier League,DF,1365,0.430


In [22]:
explain_similarity("Nampalys Mendy", "2021/22", top_n=8, n_features=8)

,player,most_similar_features
0,Idrissa Gana Gueye,"CPA_per90, CrsPA_per90, Goals_per90, SoT%, G/S..."
1,Eric García,"Assists_per90, CrsPA_per90, Goals_per90, SoT_p..."
2,Juan Bernat,"Assists_per90, Int_per90, Goals_per90, SoT_per..."
3,Nabil Bentaleb,"Assists_per90, CPA_per90, CrsPA_per90, TB_per9..."
4,Sofyan Amrabat,"Assists_per90, CrsPA_per90, TB_per90, AerWon%,..."
5,Trevoh Chalobah,"CPA_per90, CrsPA_per90, Touches_per90, PasTotA..."
6,Jérôme Boateng,"CPA_per90, Pas3rd_per90, Goals_per90, G/Sh, Cr..."
7,Daniel Amartey,"Assists_per90, CPA_per90, Goals_per90, CrsPA_p..."


### Nampalys Mendy (2021/22)

For Nampalys Mendy, we would expect a more classic defensive midfielder profile. His role is focused on defensive support, positioning and simple ball passing rather than controlling momentum or progressing the ball.

The results generally match this idea. Players such as Idrissa Gueye, Nabil Bentaleb and Sofyan Amrabat are typical defensive midfielders whose main tasks include ball recovery, protecting the defence and maintaining midfield balance.

It is also interesting that several defenders appear in the similarity list, including Eric García, Trevoh Chalobah and Daniel Amartey. This likely reflects a similar statistical profile based on defensive actions and safer passing rather than creative or attacking contribution.

In [23]:
find_similar("Conor Coady", "2021/22", top_n=8)

,Player,Season,Squad,Comp,Pos,Min,similarity
445,Wout Faes,2021/22,Reims,Ligue 1,DF,3330,0.869
884,Ricardo Mangas,2021/22,Bordeaux,Ligue 1,DF,2353,0.856
780,Víctor Laguardia,2021/22,Alavés,La Liga,DF,3042,0.836
1392,Ismaël Traoré,2021/22,Angers,Ligue 1,DF,3215,0.830
969,Stefan Mitrovi?,2021/22,Getafe,La Liga,DF,2667,0.824
394,Gabriel Dos Santos,2021/22,Arsenal,Premier League,DF,3063,0.819
1300,Milan kriniar,2021/22,Inter,Serie A,DF,3150,0.794
1219,Amir Rrahmani,2021/22,Napoli,Serie A,DF,2927,0.792


In [24]:
find_similar("Rúben Dias", "2021/22", top_n=8)

,Player,Season,Squad,Comp,Pos,Min,similarity
788,Aymeric Laporte,2021/22,Manchester City,Premier League,DF,2828,0.881
915,Joël Matip,2021/22,Liverpool,Premier League,DF,2790,0.769
1290,Thiago Silva,2021/22,Chelsea,Premier League,DF,2650,0.727
218,Duje ?aleta-Car,2021/22,Marseille,Ligue 1,DF,2057,0.718
730,Presnel Kimpembe,2021/22,Paris S-G,Ligue 1,DF,2576,0.686
23,Manuel Akanji,2021/22,Dortmund,Bundesliga,DF,2261,0.667
934,Facundo Medina,2021/22,Lens,Ligue 1,DF,2593,0.662
1130,Gerard Piqué,2021/22,Barcelona,La Liga,DF,2092,0.617


### Centre-back profiles: Conor Coady vs Rúben Dias

Both examples represent centre-backs, but the groups of similar players suggest two slightly different defensive profiles.

For Conor Coady, many of the most similar players come from teams that are not dominant in possession, such as Reims, Bordeaux, Alavés, Angers or Getafe. In these teams centre-backs are often required to defend more frequently and focus on positioning, clearances and defensive organisation rather than on initiating attacks.

At the same time, players such as Gabriel (Arsenal), Škriniar (Inter) and Rrahmani (Napoli) also appear in the similarity list. While these teams are stronger and more competitive, the defenders themselves still represent a more traditional centre-back profile focused on defending and stability rather than heavy involvement in build-up play.

In contrast, the players appearing in the Rúben Dias example mostly come from teams that dominate possession and control the tempo of the game. In these systems centre-backs are expected not only to defend but also to actively participate in build-up play and ball progression.

This difference highlights how even a simple model can distinguish between two common centre-back archetypes: more traditional defensive centre-backs and ball-playing centre-backs who are heavily involved in possession and progression.

In [30]:
find_similar("Federico Valverde", "2022/23", top_n=8)

,Player,Season,Squad,Comp,Pos,Min,similarity
2003,Teun Koopmeiners,2022/23,Atalanta,Serie A,MF,1835,0.940
1884,Vincenzo Grifo,2022/23,Freiburg,Bundesliga,FWMF,1289,0.931
2127,Brais Méndez,2022/23,Real Sociedad,La Liga,MF,1451,0.926
2077,James Maddison,2022/23,Leicester City,Premier League,MFFW,1273,0.889
2010,Andrej Kramari?,2022/23,Hoffenheim,Bundesliga,MFFW,1365,0.886
2185,Christopher Nkunku,2022/23,RB Leipzig,Bundesliga,FWMF,1282,0.878
2059,Ademola Lookman,2022/23,Atalanta,Serie A,FWMF,1299,0.876
1591,Nicolò Barella,2022/23,Inter,Serie A,MF,1609,0.869


In [29]:
explain_similarity("Federico Valverde", "2022/23", top_n=8, n_features=8)

,player,most_similar_features
0,Teun Koopmeiners,"Assists_per90, CPA_per90, SCA_per90, TouAttPen..."
1,Vincenzo Grifo,"CarPrgDist_per90, G/Sh, CrsPA_per90, TouAtt3rd..."
2,Brais Méndez,"PasAss_per90, TouAttPen_per90, CPA_per90, Touc..."
3,James Maddison,"GCA_per90, CarProg_per90, PasTotAtt_per90, Tou..."
4,Andrej Kramari?,"G/Sh, Shots_per90, TouMid3rd_per90, Goals_per9..."
5,Christopher Nkunku,"AerWon_per90, PPA_per90, PasTotPrgDist_per90, ..."
6,Ademola Lookman,"TouAtt3rd_per90, PasTotPrgDist_per90, Blocks_p..."
7,Nicolò Barella,"Blocks_per90, TouMid3rd_per90, Int_per90, TouA..."


### Federico Valverde (2022/23)

This example is particularly interesting because Federico Valverde is a very versatile player. Because of this flexibility, I was especially curious to see what kind of similar players the model would return. The same curiosity applies to the next example with Marcos Llorente, another very multi-positional and verstile player.

The results show a mix of midfielders and attacking players. Names like Koopmeiners, Maddison or Barella represent different types of midfield roles, while players like Nkunku, Kramarić or Lookman are more attacking profiles. This suggests that the model captures Valverde as a player who contributes both to progression and attacking involvement rather than fitting into a single traditional midfield role.

The similarity explanation also shows overlapping features related to attacking contribution, progression and activity in advanced areas of the pitch. This combination of qualities helps explain why Valverde appears statistically close to players with both midfield and attacking profiles.

Overall, the results reflect Valverde’s hybrid playing style. Instead of matching him only with classic central midfielders, the model identifies players who share a similar mix of progression, offensive activity and involvement across multiple phases of play.

In [31]:
find_similar("Marcos Llorente", "2021/22", top_n=8)

,Player,Season,Squad,Comp,Pos,Min,similarity
1501,Akim Zedadka,2021/22,Clermont Foot,Ligue 1,DF,3337,0.858
313,Marc Cucurella,2021/22,Brighton,Premier League,DF,3089,0.770
0,Max Aarons,2021/22,Norwich City,Premier League,DF,2881,0.708
102,Iván Balliu,2021/22,Rayo Vallecano,La Liga,DF,3147,0.678
797,Vincent Le Goff,2021/22,Lorient,Ligue 1,DF,3060,0.661
540,Andoni Gorosabel,2021/22,Real Sociedad,La Liga,DF,2289,0.658
1353,Charlie Taylor,2021/22,Burnley,Premier League,DF,2731,0.652
1435,Nacho Vidal,2021/22,Osasuna,La Liga,DF,2902,0.651


In [33]:
explain_similarity("Marcos Llorente", "2021/22", top_n=8, n_features=8)

,player,most_similar_features
0,Akim Zedadka,"CarPrgDist_per90, Goals_per90, TB_per90, PasPr..."
1,Marc Cucurella,"AerLost_per90, SoT_per90, TouAtt3rd_per90, Blo..."
2,Max Aarons,"Goals_per90, PPA_per90, G/Sh, PasTotPrgDist_pe..."
3,Iván Balliu,"SoT_per90, PasAss_per90, AerWon_per90, Clr_per..."
4,Vincent Le Goff,"Shots_per90, TouDef3rd_per90, TouMid3rd_per90,..."
5,Andoni Gorosabel,"Blocks_per90, GCA_per90, Goals_per90, G/Sh, To..."
6,Charlie Taylor,"Assists_per90, Goals_per90, G/Sh, Pas3rd_per90..."
7,Nacho Vidal,"Goals_per90, PasProg_per90, G/Sh, Int_per90, T..."


As mentioned earlier, Marcos Llorente is another very versatile player, capable of playing in different roles. Because of this flexibility, it is interesting to see that the model returns mostly full-backs and wide defenders as the most similar players.

For clarity, Marcos Llorente in the 2021/22 season played 26 games as a right wing-back, 5 as a central midfielder and 2 as a right midfielder.

Names such as Cucurella, Aarons, Vidal suggest a profile strongly linked to wide areas of the pitch, with frequent movement up and down the flank. This reflects Llorente’s role at Atlético Madrid, where he often operated in wide positions and contributed with runs, progression and defensive work rather than acting as a traditional central midfielder.